# 🔒 Task 2: Data Anonymization & Privacy

**Objective:** Anonymize the `mobile_customers` dataset to hide personal details while preserving useful information for InsightSpark's data scientists.

### 🛡️ Anonymization Techniques to Apply:

1. **Removal (Redaction)**: Remove columns with no informational value (customer_id, current_location)
2. **Masking**: Hide partial information (username, email, credit card numbers)
3. **Synthetic Data Replacement**: Replace real names, addresses, residences with fake values
4. **Noise Addition**: Add random noise to dates (date_registered, birthdate)
5. **Binning/Generalization**: Convert continuous variables to categories (age, salary)
6. **Tokenization**: Replace categorical values while preserving distribution (credit_card_provider, credit_card_expire, employer, job)

In [61]:
# ==========================================
# 🛠️ STEP 1: SETUP AND LOAD DATA
# ==========================================
import pandas as pd
import numpy as np
import hashlib
from faker import Faker
import random
from datetime import datetime, timedelta

# Initialize Faker for generating fake data
fake = Faker()

# Load the dataset
url = "https://raw.githubusercontent.com/beingAnujChaudhary/cba-insightspark-data-platform/main/data/raw/mobile_customers.xlsx"
customersData = pd.read_excel(url)

# Create a working copy
df = customersData.copy()

print(f"✅ Loaded {len(df)} customer records")
print(f"📊 Original columns: {list(df.columns)}")

✅ Loaded 10000 customer records
📊 Original columns: ['Unnamed: 0', 'customer_id', 'date_registered', 'username', 'name', 'gender', 'address', 'email', 'birthdate', 'current_location', 'residence', 'employer', 'job', 'age', 'salary', 'credit_card_provider', 'credit_card_number', 'credit_card_security_code', 'credit_card_expire']


In [62]:
# ==========================================
# 🗑️ STEP 2: REMOVE UNNECESSARY COLUMNS
# ==========================================
# Remove customer_id and current_location (no informational value)
df = df.drop(columns=['customer_id', 'current_location'])

print("✅ Removed: customer_id, current_location")
print(f"📊 Remaining columns: {list(df.columns)}")

✅ Removed: customer_id, current_location
📊 Remaining columns: ['Unnamed: 0', 'date_registered', 'username', 'name', 'gender', 'address', 'email', 'birthdate', 'residence', 'employer', 'job', 'age', 'salary', 'credit_card_provider', 'credit_card_number', 'credit_card_security_code', 'credit_card_expire']


In [63]:
# ==========================================
# 👤 STEP 3: MASK USERNAME
# ==========================================
def mask_username(username):
    """Mask username showing only first 2 and last 2 characters"""
    if pd.isna(username):
        return username
    username_str = str(username)
    if len(username_str) <= 4:
        return '*' * len(username_str)
    return username_str[:2] + '*' * (len(username_str) - 4) + username_str[-2:]

df['username'] = df['username'].apply(mask_username)

print("✅ Username masked (showing only first 2 and last 2 chars)")
print(f"Example: {df['username'].iloc[0]}")

✅ Username masked (showing only first 2 and last 2 chars)
Example: ro********an


In [64]:
# ==========================================
# 🎭 STEP 4: REPLACE NAMES WITH FAKE NAMES
# ==========================================
# Generate fake names for all records
df['name'] = [fake.name() for _ in range(len(df))]

print("✅ Names replaced with synthetic fake names")
print(f"Example fake name: {df['name'].iloc[0]}")

✅ Names replaced with synthetic fake names
Example fake name: Vanessa Taylor


In [65]:
# ==========================================
# 📧 STEP 5: MASK EMAIL ADDRESSES
# ==========================================
def mask_email(email):
    """Mask email showing only domain"""
    if pd.isna(email):
        return email
    try:
        domain = email.split('@')[1] if '@' in email else 'unknown.com'
        return f"***@{domain}"
    except:
        return "***@masked.com"

df['email'] = df['email'].apply(mask_email)

print("✅ Email addresses masked (showing only domain)")
print(f"Example: {df['email'].iloc[0]}")

✅ Email addresses masked (showing only domain)
Example: ***@hotmail.com


In [66]:
# ==========================================
# 📅 STEP 6: ADD NOISE TO DATE COLUMNS
# ==========================================
def add_date_noise(date_value, max_days=30):
    """Add random noise to dates (±30 days by default)"""
    if pd.isna(date_value):
        return date_value
    try:
        # Convert to datetime if string
        if isinstance(date_value, str):
            date_obj = datetime.strptime(date_value, '%Y-%m-%d')
        else:
            date_obj = date_value
        
        # Add random noise between -max_days and +max_days
        noise = timedelta(days=random.randint(-max_days, max_days))
        noisy_date = date_obj + noise
        
        return noisy_date.strftime('%Y-%m-%d')
    except:
        return date_value

# Apply noise to date_registered and birthdate
df['date_registered'] = df['date_registered'].apply(add_date_noise)
df['birthdate'] = df['birthdate'].apply(add_date_noise)

print("✅ Added random noise (±30 days) to date_registered and birthdate")
print(f"Example noised date: {df['date_registered'].iloc[0]}")

✅ Added random noise (±30 days) to date_registered and birthdate
Example noised date: 2021-10-12


In [67]:
# ==========================================
# 📊 STEP 7: BIN AGE AND SALARY
# ==========================================
# Bin age into categories
age_bins = [0, 18, 25, 35, 45, 55, 65, 100]
age_labels = ['<18', '18-24', '25-34', '35-44', '45-54', '55-64', '65+']
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels, right=False)

# Bin salary into brackets
salary_bins = [0, 50000, 100000, 150000, 200000, float('inf')]
salary_labels = ['<50k', '50k-100k', '100k-150k', '150k-200k', '200k+']
df['salary_bracket'] = pd.cut(df['salary'], bins=salary_bins, labels=salary_labels, right=False)

# Drop original age and salary columns
df = df.drop(columns=['age', 'salary'])

print("✅ Age binned into groups: <18, 18-24, 25-34, 35-44, 45-54, 55-64, 65+")
print("✅ Salary binned into brackets: <50k, 50k-100k, 100k-150k, 150k-200k, 200k+")

✅ Age binned into groups: <18, 18-24, 25-34, 35-44, 45-54, 55-64, 65+
✅ Salary binned into brackets: <50k, 50k-100k, 100k-150k, 150k-200k, 200k+


In [68]:
# ==========================================
# 💳 STEP 8: TOKENIZE CREDIT CARD INFO
# ==========================================
# Create tokenization mapping for credit_card_provider (preserve distribution)
unique_providers = df['credit_card_provider'].unique()
provider_tokens = {provider: f"TOKEN_{i}" for i, provider in enumerate(unique_providers)}
df['credit_card_provider'] = df['credit_card_provider'].map(provider_tokens)

# Create tokenization mapping for credit_card_expire
unique_expires = df['credit_card_expire'].unique()
expire_tokens = {expire: f"EXP_{i}" for i, expire in enumerate(unique_expires)}
df['credit_card_expire'] = df['credit_card_expire'].map(expire_tokens)

print("✅ Credit card providers tokenized")
print("✅ Credit card expiry dates tokenized")

✅ Credit card providers tokenized
✅ Credit card expiry dates tokenized


In [69]:
# ==========================================
# 🔒 STEP 9: MASK CREDIT CARD NUMBER & CVV
# ==========================================
def mask_credit_card_number(cc_number):
    """Mask credit card number showing only last 4 digits"""
    if pd.isna(cc_number):
        return cc_number
    cc_str = str(cc_number)
    if len(cc_str) <= 4:
        return '*' * len(cc_str)
    return '*' * (len(cc_str) - 4) + cc_str[-4:]

def mask_cvv(cvv):
    """Mask CVV completely"""
    if pd.isna(cvv):
        return cvv
    return '***'

df['credit_card_number'] = df['credit_card_number'].apply(mask_credit_card_number)
df['credit_card_security_code'] = df['credit_card_security_code'].apply(mask_cvv)

print("✅ Credit card numbers masked (showing only last 4 digits)")
print("✅ CVV codes completely masked (***)")

✅ Credit card numbers masked (showing only last 4 digits)
✅ CVV codes completely masked (***)


In [70]:
# ==========================================
# 💼 STEP 10: TOKENIZE EMPLOYER AND JOB
# ==========================================
# Tokenize employer (preserve distribution)
unique_employers = df['employer'].unique()
employer_tokens = {employer: f"EMP_{i}" for i, employer in enumerate(unique_employers)}
df['employer'] = df['employer'].map(employer_tokens)

# Tokenize job (preserve distribution)
unique_jobs = df['job'].unique()
job_tokens = {job: f"JOB_{i}" for i, job in enumerate(unique_jobs)}
df['job'] = df['job'].map(job_tokens)

print("✅ Employers tokenized")
print("✅ Job titles tokenized")

✅ Employers tokenized
✅ Job titles tokenized


In [71]:
# ==========================================
# 🏠 STEP 11: REPLACE ADDRESS & RESIDENCE
# ==========================================
# Generate fake addresses
df['address'] = [fake.address().replace('\n', ', ') for _ in range(len(df))]

# Generate fake residences (cities)
df['residence'] = [fake.city() for _ in range(len(df))]

print("✅ Addresses replaced with fake synthetic addresses")
print("✅ Residences replaced with fake city names")

✅ Addresses replaced with fake synthetic addresses
✅ Residences replaced with fake city names


In [73]:
# ==========================================
# ✅ STEP 12: FINAL VERIFICATION & EXPORT
# ==========================================
print("=" * 70)
print("📊 FINAL ANONYMIZED DATASET")
print("=" * 70)

# Display final columns
print(f"\n📋 Final columns: {list(df.columns)}")
print(f"\n📐 Shape: {df.shape}")
print(f"   Rows: {len(df)}")
print(f"   Columns: {len(df.columns)}")

# Show sample
print("\n🔍 Sample of anonymized data (first 3 rows):")
display(df.head(3))

# Export to CSV
import os
os.makedirs('deliverables', exist_ok=True)
output_path = 'deliverables/task2_anonymized_customers.csv'
df.to_csv(output_path, index=False)

print(f"\n🎉 SUCCESS!")
print(f"✅ Anonymized dataset exported to: {output_path}")
print(f"\n🛡️ Summary of anonymization techniques applied:")
print("   1. ✅ Removed: customer_id, current_location")
print("   2. ✅ Masked: username (partial)")
print("   3. ✅ Replaced: name (fake names)")
print("   4. ✅ Masked: email (domain only)")
print("   5. ✅ Noised: date_registered, birthdate (±30 days)")
print("   6. ✅ Binned: age → age_group, salary → salary_bracket")
print("   7. ✅ Tokenized: credit_card_provider, credit_card_expire")
print("   8. ✅ Masked: credit_card_number, credit_card_security_code")
print("   9. ✅ Tokenized: employer, job")
print("   10. ✅ Replaced: address, residence (fake values)")

📊 FINAL ANONYMIZED DATASET

📋 Final columns: ['Unnamed: 0', 'date_registered', 'username', 'name', 'gender', 'address', 'email', 'birthdate', 'residence', 'employer', 'job', 'credit_card_provider', 'credit_card_number', 'credit_card_security_code', 'credit_card_expire', 'age_group', 'salary_bracket']

📐 Shape: (10000, 17)
   Rows: 10000
   Columns: 17

🔍 Sample of anonymized data (first 3 rows):


,Unnamed: 0,date_registered,username,name,gender,address,email,birthdate,residence,employer,job,credit_card_provider,credit_card_number,credit_card_security_code,credit_card_expire,age_group,salary_bracket
0,0,2021-10-12,ro********an,Vanessa Taylor,M,"32500 Hayes Walks Suite 063, Myersside, UT 37950",***@hotmail.com,1978-02-14,Franklinmouth,EMP_0,JOB_0,TOKEN_0,**********9846,***,EXP_0,45-54,50k-100k
1,1,2019-07-27,eg***ia,Pamela Ballard,F,"5541 Valencia Bypass Apt. 528, Denisefurt, NJ ...",***@hotmail.com,1970-11-28,Carlmouth,EMP_1,JOB_1,TOKEN_1,************5979,***,EXP_1,35-44,50k-100k
2,2,2019-10-13,tu*******an,Emma Hanson,M,"8942 Jones Throughway, Port Janetfurt, NE 22793",***@gmail.com,2009-04-16,West Cindy,EMP_2,JOB_2,TOKEN_2,***************2247,***,EXP_2,45-54,200k+



🎉 SUCCESS!
✅ Anonymized dataset exported to: deliverables/task2_anonymized_customers.csv

🛡️ Summary of anonymization techniques applied:
   1. ✅ Removed: customer_id, current_location
   2. ✅ Masked: username (partial)
   3. ✅ Replaced: name (fake names)
   4. ✅ Masked: email (domain only)
   5. ✅ Noised: date_registered, birthdate (±30 days)
   6. ✅ Binned: age → age_group, salary → salary_bracket
   7. ✅ Tokenized: credit_card_provider, credit_card_expire
   8. ✅ Masked: credit_card_number, credit_card_security_code
   9. ✅ Tokenized: employer, job
   10. ✅ Replaced: address, residence (fake values)
